<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/cuda_torch_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load model weights to the GPU
model = model.to(device)

inputs = torch.randn(5,10)

# Copy inputs data from CPU to GPU
inputs = inputs.to(device)

# Launch GPU kernels (handled automatically by PyTorch)
outputs = model(inputs)

# Copy results back from GPU to CPU if needed
outputs_cpu = outputs.cpu()

print(outputs_cpu)

tensor([[-0.2556, -1.0159],
        [-0.3767, -0.8079],
        [-0.4519, -0.1576],
        [ 0.6600, -0.1926],
        [ 0.2837, -0.9882]], grad_fn=<ToCopyBackward0>)


In [ ]:
!nvidia-smi

Thu Jun 18 17:42:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P0             30W /   70W |     139MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile cuda_example.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void myKernel(float *input, float *output) {
    int idx = threadIdx.x;
    output[idx] = input[idx] * 2;
}

int main() {
    printf("Program started\n");

    float h_input[10];
    float h_output[10];

    for(int i = 0; i < 10; i++)
        h_input[i] = i;

    float *d_input, *d_output;

    printf("Allocating GPU memory\n");
    cudaMalloc(&d_input, 10 * sizeof(float));
    cudaMalloc(&d_output, 10 * sizeof(float));

    printf("Copying data to GPU\n");
    cudaMemcpy(d_input, h_input, 10 * sizeof(float), cudaMemcpyHostToDevice);

    printf("Launching kernel\n");
    myKernel<<<1,10>>>(d_input, d_output);
    cudaDeviceSynchronize();

    printf("Copying result back to CPU\n");
    cudaMemcpy(h_output, d_output, 10 * sizeof(float), cudaMemcpyDeviceToHost);

    printf("Results:\n");
    for(int i = 0; i < 10; i++)
        printf("%f\n", h_output[i]);

    cudaFree(d_input);
    cudaFree(d_output);

    printf("Program finished\n");

    return 0;
}

Overwriting cuda_example.cu


In [ ]:
!nvcc cuda_example.cu -o cuda_example
!./cuda_example

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Program started
Allocating GPU memory
Copying data to GPU
Launching kernel
Copying result back to CPU
Results:
0.000000
2.000000
4.000000
6.000000
8.000000
10.000000
12.000000
14.000000
16.000000
18.000000
Program finished
